# Notebook 01: Dataset Understanding

## Objective

The purpose of this notebook is to understand the PaySim financial transaction dataset before designing the data pipeline.

This notebook answers the following questions:

- What is the size of the dataset?
- What columns are available?
- What are the data types?
- What business entity does each column represent?
- Are there missing values?
- Are there duplicate records?
- What are the different transaction types?
- How imbalanced is the fraud data?
- What potential data quality issues exist?
- What transformations should be implemented in later pipeline stages?

This notebook is exploratory only. No data modifications are performed.

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

In [4]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "PS_20174392719_1491204439457_log.csv"

DATA_PATH

WindowsPath('c:/Projects/paysim-financial-data-pipeline/data/raw/PS_20174392719_1491204439457_log.csv')

In [5]:
df = pd.read_csv(DATA_PATH)

In [6]:
print(f"Rows : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Rows : 6,362,620
Columns : 11


In [7]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,"9,839.64",C1231006815,"170,136.00","160,296.36",M1979787155,0.00,0.00,0,0
1,1,PAYMENT,"1,864.28",C1666544295,"21,249.00","19,384.72",M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,"21,182.00",0.00,1,0
4,1,PAYMENT,"11,668.14",C2048537720,"41,554.00","29,885.86",M1230701703,0.00,0.00,0,0


In [8]:
df.sample(10, random_state=42)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
3737323,278,CASH_IN,"330,218.42",C632336343,"20,866.00","351,084.42",C834976624,"452,419.57","122,201.15",0,0
264914,15,PAYMENT,"11,647.08",C1264712553,"30,370.00","18,722.92",M215391829,0.00,0.00,0,0
85647,10,CASH_IN,"152,264.21",C1746846248,"106,589.00","258,853.21",C1607284477,"201,303.01","49,038.80",0,0
5899326,403,TRANSFER,"1,551,760.63",C333676753,0.00,0.00,C1564353608,"3,198,359.45","4,750,120.08",0,0
2544263,206,CASH_IN,"78,172.30",C813403091,"2,921,331.58","2,999,503.88",C1091768874,"415,821.90","337,649.60",0,0
3494160,259,PAYMENT,915.13,C2002954533,0.00,0.00,M290849763,0.00,0.00,0,0
2331654,188,CASH_OUT,"20,603.87",C813757373,0.00,0.00,C823291717,"558,068.66","578,672.53",0,0
1414955,139,CASH_OUT,"58,605.72",C1850864812,0.00,0.00,C618657299,"585,494.94","644,100.66",0,0
2938135,230,PAYMENT,"4,865.11",C886849972,0.00,0.00,M623175144,0.00,0.00,0,0
6133806,544,CASH_OUT,"118,131.63",C390714641,0.00,0.00,C366360355,"8,131,691.35","8,476,246.86",0,0


In [9]:
pd.DataFrame(
    {
        "Column": df.columns,
        "Datatype": df.dtypes.astype(str),
        "Missing Values": df.isna().sum().values
    }
)

,Column,Datatype,Missing Values
step,step,int64,0
type,type,str,0
amount,amount,float64,0
nameOrig,nameOrig,str,0
oldbalanceOrg,oldbalanceOrg,float64,0
newbalanceOrig,newbalanceOrig,float64,0
nameDest,nameDest,str,0
oldbalanceDest,oldbalanceDest,float64,0
newbalanceDest,newbalanceDest,float64,0
isFraud,isFraud,int64,0


In [16]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [17]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
step,"6,362,620.00",243.40,142.33,1.00,156.00,239.00,335.00,743.00
amount,"6,362,620.00","179,861.90","603,858.23",0.00,"13,389.57","74,871.94","208,721.48","92,445,516.64"
oldbalanceOrg,"6,362,620.00","833,883.10","2,888,242.67",0.00,0.00,"14,208.00","107,315.18","59,585,040.37"
newbalanceOrig,"6,362,620.00","855,113.67","2,924,048.50",0.00,0.00,0.00,"144,258.41","49,585,040.37"
oldbalanceDest,"6,362,620.00","1,100,701.67","3,399,180.11",0.00,0.00,"132,705.66","943,036.71","356,015,889.35"
newbalanceDest,"6,362,620.00","1,224,996.40","3,674,128.94",0.00,0.00,"214,661.44","1,111,909.25","356,179,278.92"
isFraud,"6,362,620.00",0.00,0.04,0.00,0.00,0.00,0.00,1.00
isFlaggedFraud,"6,362,620.00",0.00,0.00,0.00,0.00,0.00,0.00,1.00


In [18]:
memory_mb = df.memory_usage(deep=True).sum() / 1024**2

print(f"Memory Usage : {memory_mb:.2f} MB")

Memory Usage : 706.22 MB


In [19]:
missing = df.isna().sum()

missing

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [20]:
duplicates = df.duplicated().sum()

duplicates

np.int64(0)

In [21]:
for col in df.columns:
    print(col)
    print(df[col].nunique())
    print("-"*40)

step
743
----------------------------------------
type
5
----------------------------------------
amount
5316900
----------------------------------------
nameOrig
6353307
----------------------------------------
oldbalanceOrg
1845844
----------------------------------------
newbalanceOrig
2682586
----------------------------------------
nameDest
2722362
----------------------------------------
oldbalanceDest
3614697
----------------------------------------
newbalanceDest
3555499
----------------------------------------
isFraud
2
----------------------------------------
isFlaggedFraud
2
----------------------------------------


In [22]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [23]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [24]:
df["isFlaggedFraud"].value_counts()

isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64

In [25]:
df["isFlaggedFraud"].value_counts()

isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64

In [26]:
df["nameDest"].head()

0    M1979787155
1    M2044282225
2     C553264065
3      C38997010
4    M1230701703
Name: nameDest, dtype: str

In [27]:
df["nameOrig"].str.startswith("C").value_counts()

nameOrig
True    6362620
Name: count, dtype: int64

In [28]:
df["step"].describe()

count   6,362,620.00
mean          243.40
std           142.33
min             1.00
25%           156.00
50%           239.00
75%           335.00
max           743.00
Name: step, dtype: float64

In [29]:
df["step"].describe()

count   6,362,620.00
mean          243.40
std           142.33
min             1.00
25%           156.00
50%           239.00
75%           335.00
max           743.00
Name: step, dtype: float64

In [30]:
df["amount"].describe(percentiles=[0.25,0.5,0.75,0.9,0.95,0.99])

count    6,362,620.00
mean       179,861.90
std        603,858.23
min              0.00
25%         13,389.57
50%         74,871.94
75%        208,721.48
90%        365,423.31
95%        518,634.20
99%      1,615,979.47
max     92,445,516.64
Name: amount, dtype: float64

In [31]:
df[df["amount"]>200000].shape

(1673570, 11)

## Business Interpretation

Each record represents one financial transaction.

Entities:

Origin Customer

↓

Transaction

↓

Destination Customer / Merchant

Transaction Types

- PAYMENT
- CASH_IN
- CASH_OUT
- TRANSFER
- DEBIT

Fraud Label

0 = Legitimate

1 = Fraudulent

Time

step represents an hourly timestamp.

## Data Engineering Observations

1. Source file contains over 6 million records.

2. Dataset easily fits into memory using Pandas but Spark will be used for scalability.

3. No obvious missing values.

4. Fraud represents a very small percentage of transactions.

5. Customer IDs and Merchant IDs are stored as strings.

6. step can later be converted into derived fields such as:

- day
- hour
- week

7. Balance columns may introduce target leakage for fraud modeling according to the dataset documentation, but they should still be preserved in the raw and curated layers for audit and analytical purposes. Any fraud-model feature table will explicitly exclude them.

8. Bronze Layer

Store raw data exactly as received.

9. Silver Layer

Standardize names

Derive timestamps

Validate schema

10. Gold Layer

Customer summary

Daily transaction summary

Fraud summary

Feature table

## Engineering Decisions

Bronze

Store raw transaction.

Silver

Clean transaction.

Gold

Analytics tables.

Storage Format

Parquet.

Processing Engine

PySpark.

Serving Layer

PostgreSQL.

Pipeline

Airflow.

Deployment

Docker.

Next Notebook

Notebook 02

Data Profiling and Data Quality Assessment

Objectives

- Schema validation

- Business rule validation

- Null analysis

- Duplicate analysis

- Outlier analysis

- Data Quality Report

In [32]:
profile = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().sum().values / len(df) * 100),
    "unique_count": [df[col].nunique() for col in df.columns]
})

profile

,column,dtype,missing_count,missing_pct,unique_count
step,step,int64,0,0.00,743
type,type,str,0,0.00,5
amount,amount,float64,0,0.00,5316900
nameOrig,nameOrig,str,0,0.00,6353307
oldbalanceOrg,oldbalanceOrg,float64,0,0.00,1845844
newbalanceOrig,newbalanceOrig,float64,0,0.00,2682586
nameDest,nameDest,str,0,0.00,2722362
oldbalanceDest,oldbalanceDest,float64,0,0.00,3614697
newbalanceDest,newbalanceDest,float64,0,0.00,3555499
isFraud,isFraud,int64,0,0.00,2


In [33]:
profile["cardinality_pct"] = (
    profile["unique_count"] /
    len(df)
    *100
).round(2)

profile

,column,dtype,missing_count,missing_pct,unique_count,cardinality_pct
step,step,int64,0,0.00,743,0.01
type,type,str,0,0.00,5,0.00
amount,amount,float64,0,0.00,5316900,83.56
nameOrig,nameOrig,str,0,0.00,6353307,99.85
oldbalanceOrg,oldbalanceOrg,float64,0,0.00,1845844,29.01
newbalanceOrig,newbalanceOrig,float64,0,0.00,2682586,42.16
nameDest,nameDest,str,0,0.00,2722362,42.79
oldbalanceDest,oldbalanceDest,float64,0,0.00,3614697,56.81
newbalanceDest,newbalanceDest,float64,0,0.00,3555499,55.88
isFraud,isFraud,int64,0,0.00,2,0.00


## Engineering Observations

The dataset contains no missing values.

The schema is stable.

Identifier columns (nameOrig and nameDest) exhibit high cardinality.

Transaction type and fraud indicators have low cardinality.

Balance columns have millions of distinct values.

The amount column behaves as a continuous variable.

The dataset is well suited for distributed processing because of its large number of records and high-cardinality identifiers.